# Discord message scraper

This notebook contains a small script to fetch all messages from a specific user in a guild.

Notes: You must enable the MESSAGE_CONTENT intent for your bot in the Discord Developer Portal and invite the bot with permissions to read message history. Provide `DISCORD_TOKEN`, `GUILD_ID`, and `TARGET_USER_ID` via environment variables or a `.env` file. Respect privacy and get consent before scraping messages.

In [1]:
# Token and env checker (does NOT connect the bot)
from dotenv import load_dotenv
import os

load_dotenv()
TOKEN = os.getenv('DISCORD_TOKEN')
GUILD_ID = os.getenv('GUILD_ID')
TARGET_USER_ID = os.getenv('TARGET_USER_ID')

if TOKEN:
    masked = TOKEN[:6] + '...' + TOKEN[-6:]
    print('DISCORD_TOKEN found, masked:', masked)
else:
    print('DISCORD_TOKEN not found. Put your bot token in a .env file or environment variable.')

print('GUILD_ID:', GUILD_ID or 'not set')
print('TARGET_USER_ID:', TARGET_USER_ID or 'not set')

print('\nTo run the scraper cell: make sure `requirements.txt` is installed, then run the scraper code cell in this notebook. The scraper will connect to Discord using the token from .env and collect messages to messages.jsonl')

DISCORD_TOKEN found, masked: OTE4ND...Bwx_nw
GUILD_ID: 451749529339953152
TARGET_USER_ID: 324653924416094212

To run the scraper cell: make sure `requirements.txt` is installed, then run the scraper code cell in this notebook. The scraper will connect to Discord using the token from .env and collect messages to messages.jsonl


In [ ]:
# Discord message scraper with resume support
# Usage: create a .env with DISCORD_TOKEN, GUILD_ID, TARGET_USER_ID or set them in environment variables.
import os
import json
import asyncio
from dotenv import load_dotenv
import discord

load_dotenv()
TOKEN = os.getenv('DISCORD_TOKEN')
GUILD_ID = int(os.getenv('GUILD_ID') or 0)
TARGET_USER_ID = int(os.getenv('TARGET_USER_ID') or 0)
OUTPUT = os.getenv('OUTPUT_FILE', 'messages.jsonl')
STATE_FILE = os.getenv('STATE_FILE', '.scrape_state.json')

# Intents: message_content is required to read message content (privileged intent).
intents = discord.Intents.default()
intents.message_content = True
intents.guilds = True
intents.messages = True
intents.members = True

client = discord.Client(intents=intents)


def load_state():
    try:
        with open(STATE_FILE, 'r', encoding='utf-8') as sf:
            return json.load(sf)
    except Exception:
        return {}


def save_state(state):
    try:
        tmp = STATE_FILE + '.tmp'
        with open(tmp, 'w', encoding='utf-8') as sf:
            json.dump(state, sf)
        os.replace(tmp, STATE_FILE)
    except Exception as e:
        print('Warning: failed to save state:', e)


@client.event
async def on_ready():
    print(f'Logged in as {client.user} (ID: {client.user.id})')
    guild = client.get_guild(GUILD_ID)
    if guild is None:
        print('Guild not found. Make sure the bot is in the guild and GUILD_ID is correct.')
        await client.close()
        return

    user_id = TARGET_USER_ID
    count = 0

    state = load_state()

    # Open output in append mode so we don't overwrite previous results when resuming
    with open(OUTPUT, 'a', encoding='utf-8') as f:
        for channel in guild.text_channels:
            last_id = state.get(str(channel.id))
            after = None
            if last_id:
                try:
                    after = discord.Object(id=int(last_id))
                except Exception:
                    after = None

            try:
                # Iterate through the full history or only messages after last saved id.
                async for msg in channel.history(limit=None, oldest_first=True, after=after):
                    if msg.author and msg.author.id == user_id:
                        payload = {
                            'id': msg.id,
                            'content': msg.content,
                            'author': {'id': msg.author.id, 'name': str(msg.author)},
                            'channel': {'id': channel.id, 'name': channel.name},
                            'created_at': msg.created_at.isoformat() if msg.created_at else None,
                            'attachments': [a.url for a in msg.attachments]
                        }
                        f.write(json.dumps(payload, ensure_ascii=False) + '\n')
                        count += 1
                        # update last processed id for this channel
                        state[str(channel.id)] = msg.id

                        # checkpoint periodically
                        if count % 50 == 0:
                            save_state(state)
                # small pause between channels to be polite to the API
                await asyncio.sleep(1)
            except Exception as e:
                print(f'Skipping channel {channel.name} due to {e}')

    # final save
    save_state(state)
    print(f'Done. Collected {count} new messages to {OUTPUT}')
    await client.close()


def start_client():
    """Start the Discord client safely in script or notebook.
    Call start_client() in a notebook cell to start in background, or run as a script.
    """
    if not TOKEN or not GUILD_ID or not TARGET_USER_ID:
        print('Set DISCORD_TOKEN, GUILD_ID, TARGET_USER_ID environment variables (or provide a .env).')
        return

    # Detect notebook environment
    try:
        get_ipython  # type: ignore
        in_notebook = True
    except NameError:
        in_notebook = False

    if in_notebook:
        try:
            import nest_asyncio
            nest_asyncio.apply()
        except Exception:
            print('Warning: nest_asyncio not available — install it with `pip install nest_asyncio` to run in notebooks.')

        loop = asyncio.get_event_loop()
        if not loop.is_running():
            loop.run_until_complete(client.start(TOKEN))
        else:
            loop.create_task(client.start(TOKEN))
            print('Client started in background task; it will run until you stop it or the kernel stops.')
    else:
        client.run(TOKEN)


# Start when executed as script or call start_client() in notebook
if __name__ == '__main__':
    start_client()


RuntimeError: asyncio.run() cannot be called from a running event loop